# Language Detection Model

This project focuses on Language Detection, which is a multi-class classification task in the field of NLP. Our goal is to train a model to automatically identify the language of a given text.

While identifying a language might seems obvious to us human beings, it is a complex challenge for a computer. The difficulty lies in the fact that many languages share the exact same alphabet. For example, English, French, and Spanish all use Latin letters, while several Middle Eastern and Asian languages might use similar-looking characters. A machine cannot "read" like we do. instead, it must uncover hidden patterns and statistical fingerprints, such as the frequency of specific character combinations (n-grams) or unique word structures that define each language.

We are using a [Kaggle dataset that contains 10,000 sentences across 17 different languages](https://www.kaggle.com/datasets/basilb2s/language-detection), including English, Arabic, Spanish, and others. To evaluate our success, we will use the Macro-Average F1 score, which ensures that the model is accurate across all languages, regardless of how many sentences they have in the dataset.

## Project Members

| Name | Last 4 Digits of ID |
| :--- | :--- |
| Almog S | 5890 |
| Eliad B | 6279 |

## Import Dependencies

In [17]:
import pandas as pd
import numpy as np
import sys
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

text_col = 'Text'
label_col = 'Language'

def load_dataset(file_path):
    return pd.read_csv(file_path, encoding='utf-8')

def get_features(raw, vectorized_data, row_idx):
    
    row = vectorized_data[row_idx]
    nonzero_indices = row.indices
    tfidf_weights = row.data
    
    all_feature_names = vectorizer.get_feature_names_out()
    specific_feature_names = [all_feature_names[i] for i in nonzero_indices]
    table = pd.DataFrame([tfidf_weights], columns=specific_feature_names)
    display(table)

def predict_language(input_text):
    # Vectorize the new input using the same vectorizer
    vector = vectorizer.transform([input_text])
    prediction = knn.predict(vector)
    return prediction[0]

## Loading the dataset

Our dataset is stored in a CSV file that contains 2 columns: the **text** and its corresponding **language** label. A DataFrame is the perfect tool for organizing the data and the labels into a clean, tabular structure that is easy to manipulate.

In [18]:
df = load_dataset('dataset.csv')
print(f"{df.shape[0]} sentences have been processed.")
df.head()

10337 sentences have been processed.


,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English


## Data Splitting

First, we split the data into train set and test set. We chose to split the data into 80% training and 20% testing.

In [19]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df[text_col], df[label_col], test_size=0.2, random_state=42)

train_preview = pd.concat([X_train_raw, y_train], axis=1)
train_preview.columns = ['Text Sample', 'Language']

test_preview = pd.concat([X_test_raw, y_test], axis=1)
test_preview.columns = ['Text Sample', 'Language']

print("### Table 1: Train Set Snapshot (First 5 Rows)")
display(train_preview.head())
print("### Table 2: Test Set Snapshot (First 5 Rows)")
display(test_preview.head())

### Table 1: Train Set Snapshot (First 5 Rows)


,Text Sample,Language
5967,τώρα αργότερα η Μέλι και ο Τέρι έσπασαν αντίο ...,Greek
3570,Améliorez-le ou discutez-en.,French
7607,"non ne vale la pena, personalmente amo la fras...",Italian
3080,tente copiar minha pronúncia exatamente inclui...,Portugeese
828,If the complexity of the model is increased in...,English


### Table 2: Test Set Snapshot (First 5 Rows)


,Text Sample,Language
6662,"И с этими словами она села в его карету, и, да...",Russian
7362,Sistemi di tipo probabilistico erano invasi di...,Italian
765,Machine learning involves computers discoverin...,English
6192,Несколько языковых версий опубликовали подборк...,Russian
562,"[225] In May 2014, Wikimedia Foundation named ...",English


## Feature Engineering

In this step, we transform the raw text into a numerical format that the machine learning algorithm can understand. We use the **TF-IDF Vectorizer with Character N-grams** (ranging from 1 to 3 characters): Instead of looking at complete words, the model breaks down each sentence into small sequences of single letters, pairs, and triplets. This is very effective for language detection because every language has a unique statistical "fingerprint" based on how often specific character combinations appear. This method allows the model to differentiate between languages that share the same alphabet and stay accurate even when there are typos or unfamiliar words.

**How does TF-IDF work?**

The value in each cell is calculated using this formula:

$$Score = TF \times IDF$$

Where:

$$TF = \frac{\text{Number of times the sequence appears in the row}}{\text{Total number of sequences in that row}}$$

$$IDF = \log \left( \frac{\text{Total number of rows in the DataFrame}}{\text{Number of rows containing the sequence}} \right)$$

In other words, $TF$ indicates how many times a specific letter sequence appears in a single sentence, while $IDF$ indicates how rare this sequence is across the entire dataset.

**The higher the score, the more important that character sequence is for identifying the language.**

In [20]:
# analyzer='char' tells the model to look at characters, not words.
# ngram_range=(1, 3) looks at single letters, pairs, and triplets.
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3))
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

# X_train and X_test contain over 60,000 columns!
print(X_train.shape)
print(X_test.shape)

(8269, 62628)
(2068, 62628)


**Why are there over 60,000 columns?**

The high number of features comes from the 17 different languages in our dataset. Since these languages use different alphabets (like Arabic, Greek, and Latin), every unique combination of 3 letters creates a new column in our matrix. With 10,000 sentences, the number of unique combinations grows very quickly.

Below are examples of the first 5 features of 2 sentences.

In [21]:
row_idx = 0
print(f"Training Sample:\nOriginal Sentence: \"{X_train_raw.iloc[row_idx]}\"\n")
get_features(X_train_raw, X_train, row_idx)
print("=" * 40);
print(f"Test Sample:\nOriginal Sentence: \"{X_train_raw.iloc[row_idx]}\"\n")
get_features(X_test_raw, X_test, row_idx)

Training Sample:
Original Sentence: "τώρα αργότερα η Μέλι και ο Τέρι έσπασαν αντίο στον παλιό τους φίλο και πήγαν να χαμογελούν ο ένας στον άλλο κρυφά εκείνο το βράδυ, τόσο μητέρα όσο και κόρη."



,τ,ώ,ρ,α,,γ,ό,ε,η,μ,...,μη,μητ,ητέ,έρα,α ό,όσ,κό,κόρ,όρη,ρη.
0,0.219693,0.0254,0.178591,0.322737,0.145366,0.070328,0.116456,0.086342,0.070405,0.066371,...,0.034106,0.037986,0.037089,0.036695,0.035673,0.039742,0.038505,0.039742,0.039742,0.045908


Test Sample:
Original Sentence: "τώρα αργότερα η Μέλι και ο Τέρι έσπασαν αντίο στον παλιό τους φίλο και πήγαν να χαμογελούν ο ένας στον άλλο κρυφά εκείνο το βράδυ, τόσο μητέρα όσο και κόρη."



,,а,ан,б,бу,в,в,вн,во,г,...,ьн,ьно,ью,"ью,",э,эт,эти,ю,"ю,","ю,"
0,0.148328,0.021124,0.026189,0.018894,0.027459,0.079946,0.037832,0.029662,0.046786,0.020344,...,0.021775,0.023135,0.026934,0.031784,0.018748,0.01972,0.027276,0.018377,0.026619,0.026619


## Model Training (KNN)

Now that our text is represented as numerical vectors (TF-IDF), we can apply the **KNN algorithm**. 

The core logic of KNN is simple: to predict the language of a new sentence, the model looks at the $k$ most similar sentences in our training data.

- For a given input vector, we calculate its **Euclidean Distance** to every single vector in our training set. The distance $d$ between two points $x$ and $y$ is calculated as:$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

In our specific model, $n = 62,628$. Each distance calculation involves summing the squared differences across all 62,628 dimensions (features).

- We sort these distances in ascending order and select the $k$ closest neighbors.
- We check the labels (languages) of these $k$ neighbors. The language that appears most frequently among them is chosen as the final prediction.

**Note**: We will explore different K-values using Cross-Validation at the [Evaluation](#Evaluation) step to find the optimal hyperparameters.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5) # k = 5
knn.fit(X_train, y_train)

## Model Testing

Now, we want to check if our model is actually successful. It is important to know if the model truly "learned" how to identify languages or if the results were just a matter of luck. To do this, we will test the model on the Test Set data that the model has never seen before.

In [22]:
y_pred = knn.predict(X_test)

sample_count = 20
results_table = pd.DataFrame({'Text': X_test_raw[:sample_count], 'Actual': y_test[:sample_count], 'Predicted': y_pred[:sample_count]})
display(results_table)

,Text,Actual,Predicted
6662,"И с этими словами она села в его карету, и, да...",Russian,Russian
7362,Sistemi di tipo probabilistico erano invasi di...,Italian,Italian
765,Machine learning involves computers discoverin...,English,English
6192,Несколько языковых версий опубликовали подборк...,Russian,Russian
562,"[225] In May 2014, Wikimedia Foundation named ...",English,English
6984,lad mig komme tilbage til dig om det.,Danish,Danish
39,Earth is estimated to have formed 4.54 billion...,English,English
1597,മാതൃക (ഗണിത സമവാക്യത്തിന്റെ മാതൃക) തീരുമാനിച്ച...,Malayalam,Malayalam
6202,В декабре того же года появилось третье издание.,Russian,Russian
10241,"ದುರದೃಷ್ಟವಶಾತ್, ನಾನು ಇಲ್ಲ ಎಂದು ಹೇಳಬೇಕಾಗಿದೆ.",Kannada,Kannada


Great! our predictions match the results.

## Evaluation

This step will help us answer two key questions:

- How accurate is the model on new text, one that it has never seen before?

- How often does it make mistakes?

We will use the Accuracy score and the Macro F1-score to see the full picture and ensure the model performs well across all 17 languages.

In [16]:
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nFull Performance Report:")
print(classification_report(y_test, y_pred))

Overall Accuracy: 97.58%

Full Performance Report:
              precision    recall  f1-score   support

      Arabic       0.99      1.00      1.00       106
      Danish       0.88      0.96      0.92        73
       Dutch       0.95      0.97      0.96       111
     English       0.97      0.99      0.98       291
      French       0.98      0.98      0.98       219
      German       1.00      0.96      0.98        93
       Greek       1.00      1.00      1.00        68
       Hindi       1.00      1.00      1.00        10
     Italian       0.95      0.96      0.95       145
     Kannada       1.00      1.00      1.00        66
   Malayalam       1.00      0.99      1.00       121
  Portugeese       0.96      0.95      0.96       144
     Russian       1.00      1.00      1.00       136
     Spanish       0.97      0.95      0.96       160
    Sweedish       0.99      0.94      0.97       133
       Tamil       1.00      1.00      1.00        87
     Turkish       1.00      0

## DIY

Enter your desired text and check our model!

In [25]:
text = 'Geil das du streamst' # EDIT THIS VALUE TO PREDICT YOUR TEXT'S LANGUAGE
print(f"Input: '{text}' -> Predicted: {predict_language(text)}")

Input: 'Geil das du streamst' -> Predicted: German
